# v12 M: three interpretability tests of the self (no training)
Ideas from *Reading Model Minds*, applied to the translation models. Self, task and look-alike picks always have the **same number of neurons** (half of the smaller group, per layer).
- **Dial:** multiply the picked neurons' activity by 0, 0.5, 1.5, 2; hold them at their average; and push the decoder along the *confident minus unsure* direction with the **same push size for every group** (the fair version of the old steering test; the old unfair version is kept for comparison).
- **Broadcast:** nudge the model along one neuron's outgoing direction (same nudge size for all) and measure how much the final hidden state moves. Self neurons vs task neurons vs random directions.
- **Overlap:** find the self separately on each language's text; how much do the sets overlap compared with chance?

Models: distant 40, 41, 42 · related 13, 14, 15 · Hindi-only 45 (self found on Hindi) · mixed 45. Measured on the language each checkpoint was last trained on (Hindi for mixed).
Results: `analysis/v12/dial`, `broadcast`, `overlap` · progress: `tail -f logs/v12_M_mind.log`

In [ ]:
import os, sys, glob, json, time, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
GPU = os.environ.get("CML_GPU", "0"); LOG = "v12_M_mind"
REPO = "/workspace/shreshth"; os.chdir(REPO); sys.path.insert(0, REPO)
from IPython.display import Image, display
from huggingface_hub import HfApi, hf_hub_download
FILES, HF_REPO = [], None
for attempt in range(5):
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; FILES = api.list_repo_files(HF_REPO); break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)
if not FILES:
    print("could not list the HF repo: only checkpoints already on this machine will be used")

def have(run, ph):
    return bool(glob.glob(f"runs/{run}/ckpt/p{ph:03d}_*.pt"))

def get(run, phases):
    """download what is missing; returns the phases that are on this machine afterwards"""
    ok = []
    for ph in phases:
        if not have(run, ph):
            name = [f for f in FILES if f.startswith(f"{run}/ckpt/p{ph:03d}_")]
            if name:
                try:
                    hf_hub_download(HF_REPO, name[0], local_dir="runs")
                except Exception as e:
                    print("download failed:", run, ph, type(e).__name__, e)
        if have(run, ph):
            ok.append(ph)
        else:
            print("not available (not here, not on HF):", run, ph)
    return ok

def run(langset, script, *cmd):
    """one call of a script; output shown here AND appended to logs/<LOG>.log (watch with: tail -f logs/<LOG>.log)"""
    env = {**os.environ, "CML_LANGSET": langset}; os.makedirs("logs", exist_ok=True)
    p = subprocess.Popen([sys.executable, "-u", script, "--gpu", GPU, *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    with open(f"logs/{LOG}.log", "a") as log:
        for line in p.stdout:
            print(line, end=""); log.write(line); log.flush()
    p.wait(); print("exit code", p.returncode)

def summary(script, part, out=None):
    cmd = [sys.executable, script, "--summarize", "--part", part] + (["--out", out] if out else [])
    r = subprocess.run(cmd, capture_output=True, text=True); print(r.stdout[-14000:]); print(r.stderr[-3000:])

def show(*paths):
    for f in paths:
        if os.path.exists(f): display(Image(f))

DIST, HIN, MIX, REL, REL1 = "v4b_continual_s0", "v4b_only_hi_s0", "v6b_mixed_s0", "v8_indic_continual_s0", "v8_indic_continual_s1"
print("GPU", GPU)

In [ ]:
PLAN = [("distant", "distant", DIST, [40, 41, 42], None), ("indic", "related", REL, [13, 14, 15], None),
        ("distant", "hindi", HIN, [45], "hi"), ("distant", "mixed", MIX, [45], None)]
def go(part, plan):
    for langset, fam, r, phs, sl in plan:
        ok = get(r, phs)
        if ok:
            cmd = ["--part", part, "--family", fam, "--model", f"{fam}={r}:" + ",".join(str(p) for p in ok)]
            run(langset, "mind_v12.py", *(cmd + (["--self_langs", sl] if sl else [])))
go("dial", PLAN)
summary("mind_v12.py", "dial")
show("analysis/v12/dial/dial_gain_d_loss.png", "analysis/v12/dial/dial_steer_same_d_entropy.png")

In [ ]:
go("broadcast", PLAN)
summary("mind_v12.py", "broadcast")

In [ ]:
go("overlap", [p for p in PLAN if p[1] != "hindi"])
summary("mind_v12.py", "overlap")